# Nairobi Urban Flood Digital Twin — Model Training

Trains the flood-forecast U-Net on Colab's GPU.

**Task.** Given the previous 7 days of rainfall plus terrain, predict flood extent over the next 3 days. The input and label windows do not overlap, so the model must anticipate a storm rather than read the answer off an input channel.

**Inputs (13 channels):** 7 antecedent rainfall days + DEM, slope, TWI, HAND, built-up, permanent water.

**Runtime.** A few minutes on a T4. The whole dataset is ~1.8 MB and lives on the GPU, so there is no DataLoader, no memory-mapping, and system RAM is not involved.

Before running: set `REPO_URL` in the next cell, and enable the GPU via **Runtime -> Change runtime type -> T4 GPU**.

Read `LIMITATIONS.md` in the repo before quoting any number from this notebook in the thesis.

## 1. Environment

In [ ]:
REPO_URL = "https://github.com/eoringe/nairobi-flood-digital-twi.git"
BRANCH   = "main"

import torch, subprocess
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device      :", torch.cuda.get_device_name(0))
    print("VRAM        : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("\n!! No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")

In [ ]:
# Optional: mount Drive to persist outputs across a runtime disconnect.
# The dataset does NOT come from Drive -- it ships in the repo.
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("\nDrive mounted. Outputs will be copied there in section 5.")
else:
    print("Skipping Drive. Outputs stay in the Colab runtime and are lost on disconnect.")

## 2. Get the code and data

The dataset is committed to the repository (~1.8 MB), so cloning brings it. No upload step.

It is small because rainfall is stored as 7 scalars per sample and terrain as one shared array, rather than the earlier format's 77 channels that were 91% duplicated copies — that version was 6 GB and repeatedly exhausted RAM.

In [ ]:
import os, shutil
os.chdir('/content')
if os.path.isdir('nairobi-flood-digital-twi'):
    shutil.rmtree('nairobi-flood-digital-twi')

!git clone --branch $BRANCH --single-branch $REPO_URL 2>&1 | tail -5
os.chdir('/content/nairobi-flood-digital-twi')

!pip install -q torch numpy matplotlib
print("\nWorking directory:", os.getcwd())

## 3. Verify the dataset

In [ ]:
import numpy as np, os, json

DATASET = 'data/processed/arrays/segmentation_dataset_v2.npz'
if not os.path.exists(DATASET):
    raise FileNotFoundError(
        DATASET + " not found.\n"
        "Rebuild locally with:  python -m src.ingestion.build_segmentation_dataset_v2\n"
        "then commit, push, and re-run the clone cell."
    )

d = np.load(DATASET, allow_pickle=False)
print("Dataset OK  (%.1f MB)\n" % (os.path.getsize(DATASET) / 1e6))
print("  rain_seq :", d['rain_seq'].shape, "  antecedent rainfall, mm/day")
print("  static   :", d['static'].shape, " ", list(d['channel_names'][:6]))
print("  y        :", d['y'].shape, " dtype =", d['y'].dtype)
print()
print("  train / val / test : %d / %d / %d samples"
      % (len(d['train_idx']), len(d['val_idx']), len(d['test_idx'])))
print("  positive pixel rate: %.2f%%" % (100 * d['y'].mean()))
print("  distinct masks     : %d" % len({y.tobytes() for y in d['y']}))
print()
print("  label parameters:")
for k, v in json.loads(str(d['params'][0])).items():
    print("    %-20s %s" % (k, v))

## 4. Train

Training runs from the repository script rather than a copy pasted into this cell. Keeping the model definition in one place is deliberate — an earlier version of this notebook carried its own divergent copy of the U-Net, which is how several channel-mismatch and memory bugs survived repeated edits.

Watch **precision** and **recall** together. The loss (Focal Tversky, β > α) deliberately penalises missed floods more than false alarms, so recall rises first and precision follows. Recall pinned at 0 would mean the old all-background collapse has returned.

In [ ]:
!python -m src.models.train_segmentation_v2 \
    --epochs 60 --batch-size 16 --base 32 --lr 1e-3

## 5. Results

In [ ]:
import json, os, shutil

metrics_path = 'models/time_series/segmentation_metrics_v2.json'
m = json.load(open(metrics_path))

print("=" * 56)
print("TEST -- held-out storm seasons, never seen in training")
print("=" * 56)
for k in ('f1', 'iou', 'precision', 'recall'):
    print("  %-11s %.4f" % (k, m['test_metrics'][k]))
print("\n  best validation F1: %.4f" % m['best_val_f1'])

print("\nValidation trajectory:")
print("  %-7s %-11s %-9s %-9s %-9s" % ("epoch", "loss", "f1", "precision", "recall"))
for h in m['history'][::5]:
    print("  %-7d %-11.4f %-9.4f %-9.4f %-9.4f"
          % (h['epoch'], h['train_loss'], h['f1'], h['precision'], h['recall']))

if USE_DRIVE:
    out = '/content/drive/MyDrive/nairobi-flood-data/training-outputs'
    os.makedirs(out, exist_ok=True)
    for f in ['models/time_series/segmentation_model_v2.pth', metrics_path]:
        shutil.copy(f, out + '/' + os.path.basename(f))
    print("\nSaved to Drive:", out)

## 6. Example predictions

Generates side-by-side figures for the thesis Results chapter: terrain susceptibility, the label, and the model's predicted probability for held-out test storms.

Inspect these rather than trusting F1 alone. Predicted flooding should follow drainage lines and low-lying ground. If it looks like uniform blanket coverage, the model has learned the storm trigger but not the spatial distribution.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, torch, json

from src.models.train_segmentation_v2 import UNet, GpuDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
d = np.load('data/processed/arrays/segmentation_dataset_v2.npz', allow_pickle=False)

model = UNet(in_ch=13, base=32).to(device)
model.load_state_dict(torch.load('models/time_series/segmentation_model_v2.pth',
                                 map_location=device))
model.eval()

test = GpuDataset(d, d['test_idx'], device)
susc = d['susceptibility']
dates = d['dates'][d['test_idx']]
events = d['event_ids'][d['test_idx']]

# Pick the largest-extent storm from each distinct test event. Taking the global
# top-N instead would return consecutive days of the same storm, which look
# nearly identical and waste a figure panel.
extents = np.array([d['y'][j].mean() for j in d['test_idx']])
picks = []
for ev in sorted(set(events)):
    cand = np.where((events == ev) & (extents > 0))[0]
    if len(cand):
        picks.append(int(cand[np.argmax(extents[cand])]))
picks = sorted(picks, key=lambda i: -extents[i])[:6]
print("Showing %d storms from %d held-out seasons\n" % (len(picks), len(set(events[picks]))))

fig, axes = plt.subplots(len(picks), 3, figsize=(11, 3.1 * len(picks)))
for row, i in enumerate(picks):
    with torch.no_grad():
        x, y = test.batch(torch.tensor([i], device=device))
        prob = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    truth = y[0, 0].cpu().numpy()

    for ax, img, title, cmap in [
        (axes[row, 0], susc, 'Terrain susceptibility', 'terrain_r'),
        (axes[row, 1], truth, 'Label (extent %.1f%%)' % (100 * truth.mean()), 'Blues'),
        (axes[row, 2], prob, 'Predicted probability', 'Blues'),
    ]:
        im = ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    axes[row, 0].set_ylabel('%s\n%s' % (dates[i], events[i]), fontsize=8)

plt.tight_layout()
plt.savefig('models/time_series/example_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: models/time_series/example_predictions.png")

if USE_DRIVE:
    import shutil
    shutil.copy('models/time_series/example_predictions.png',
                '/content/drive/MyDrive/nairobi-flood-data/training-outputs/')
    print("Copied to Drive.")